# IIT414W Lab 2 — Feature Engineering + Improved Baseline

**Team:** Carlos Orellana & Mattias Morales (Group 6)  
**Course:** IIT414W — Artificial Intelligence Workshop  
**Date:** March 2026

## Objective
Build an improved baseline model for predicting F1 Top-10 finishes using leakage-safe, pre-race feature engineering, and compare fairly against Lab 1 baselines on the same temporal validation split.


In [1]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print(f"RANDOM_SEED = {RANDOM_SEED}")


RANDOM_SEED = 414


## 1. Data Pipeline (same source style as Lab 1)
We fetch race results from the Jolpica API for seasons 2022–2024 with pagination.


In [2]:
def fetch_season_results(year: int, max_retries: int = 3) -> pd.DataFrame:
    all_races = []
    offset = 0
    limit = 100

    while True:
        url = f"https://api.jolpi.ca/ergast/f1/{year}/results.json?limit={limit}&offset={offset}"

        for attempt in range(max_retries):
            try:
                resp = requests.get(url, timeout=30)
                resp.raise_for_status()
                data = resp.json()["MRData"]
                total = int(data["total"])
                races = data["RaceTable"]["Races"]
                all_races.extend(races)
                break
            except Exception:
                if attempt == max_retries - 1:
                    raise
                time.sleep(1.0)

        offset += limit
        if offset >= total:
            break
        time.sleep(0.5)

    rows = []
    for race in all_races:
        for res in race["Results"]:
            rows.append(
                {
                    "season": int(race["season"]),
                    "round": int(race["round"]),
                    "race_name": race["raceName"],
                    "circuit": race["Circuit"]["circuitId"],
                    "date": race["date"],
                    "driver": res["Driver"]["driverId"],
                    "driver_code": res["Driver"].get("code", res["Driver"]["driverId"][:3].upper()),
                    "constructor": res["Constructor"]["constructorId"],
                    "constructor_name": res["Constructor"]["name"],
                    "grid": int(res["grid"]),
                    "position": int(res["position"]) if res["position"].isdigit() else np.nan,
                    "position_text": res["positionText"],
                    "status": res["status"],
                }
            )

    return pd.DataFrame(rows)


frames = []
for year in [2022, 2023, 2024]:
    print(f"Fetching {year}...")
    frames.append(fetch_season_results(year))
    time.sleep(0.5)

results = pd.concat(frames, ignore_index=True)
results["date"] = pd.to_datetime(results["date"])

# Target from final race result
results["top10_finish"] = ((results["position"].notna()) & (results["position"] <= 10)).astype(int)

print(f"Total rows: {len(results)}")
print(results[["season", "round", "race_name", "driver", "constructor", "grid", "position", "top10_finish"]].head(3))


Fetching 2022...


Fetching 2023...


Fetching 2024...


Total rows: 1359
   season  round           race_name    driver constructor  grid  position  top10_finish
0    2022      1  Bahrain Grand Prix   leclerc     ferrari     1         1             1
1    2022      1  Bahrain Grand Prix     sainz     ferrari     3         2             1
2    2022      1  Bahrain Grand Prix  hamilton    mercedes     5         3             1


## 2. Feature Engineering (pre-race only)
We engineer 4 new features with temporal safeguards.


In [3]:
# Chronological sort for temporal features
results = results.sort_values(["date", "season", "round", "driver"]).reset_index(drop=True)

# Use a bounded fallback for DNFs in historical finish-position features
# (unknown/no finish treated as 20th for history only)
results["position_for_history"] = results["position"].fillna(20)

# 1) Lag feature: previous race finish position by driver
results["prev_finish_pos"] = (
    results.groupby("driver")["position_for_history"].shift(1)
)

# 2) Rolling aggregate: average finish over last 3 races (prior only)
results["avg_finish_pos_last3_prior"] = (
    results.groupby("driver")["position_for_history"]
    .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
)

# 3) Interaction + historical rate: driver x circuit prior top10 rate
results["driver_circuit_top10_rate_prior"] = (
    results.groupby(["driver", "circuit"])["top10_finish"]
    .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
)

# 4) Categorical encoding: constructor tier from PREVIOUS season only
records = []
for season in sorted(results["season"].unique()):
    prev_season = season - 1
    prev_rates = (
        results.loc[results["season"] == prev_season]
        .groupby("constructor")["top10_finish"]
        .mean()
    )
    if prev_rates.empty:
        continue

    pct = prev_rates.rank(method="first", pct=True)
    tier = pd.Series(
        np.where(pct > (2/3), "top", np.where(pct > (1/3), "mid", "back")),
        index=prev_rates.index,
    )

    rec = pd.DataFrame(
        {
            "season": season,
            "constructor": tier.index,
            "constructor_tier_prev_season": tier.values,
        }
    )
    records.append(rec)

if records:
    constructor_tiers = pd.concat(records, ignore_index=True)
else:
    constructor_tiers = pd.DataFrame(columns=["season", "constructor", "constructor_tier_prev_season"])

results = results.merge(constructor_tiers, on=["season", "constructor"], how="left")
results["constructor_tier_prev_season"] = results["constructor_tier_prev_season"].fillna("unknown")

feature_preview_cols = [
    "season", "round", "driver", "circuit", "constructor", "grid",
    "prev_finish_pos", "avg_finish_pos_last3_prior", "driver_circuit_top10_rate_prior",
    "constructor_tier_prev_season", "top10_finish"
]

print(results[feature_preview_cols].head(10))


   season  round           driver  circuit   constructor  grid  prev_finish_pos  avg_finish_pos_last3_prior  driver_circuit_top10_rate_prior constructor_tier_prev_season  top10_finish
0    2022      1            albon  bahrain      williams    14              NaN                         NaN                              NaN                      unknown             0
1    2022      1           alonso  bahrain        alpine     8              NaN                         NaN                              NaN                      unknown             1
2    2022      1           bottas  bahrain          alfa     6              NaN                         NaN                              NaN                      unknown             1
3    2022      1            gasly  bahrain    alphatauri    10              NaN                         NaN                              NaN                      unknown             0
4    2022      1         hamilton  bahrain      mercedes     5              NaN 

### Feature Justification (1–2 sentences each)
- **`prev_finish_pos` (lag):** Recent finishing position captures short-term driver/car form and reliability entering the next race.
- **`avg_finish_pos_last3_prior` (rolling aggregate):** A 3-race rolling average smooths race-to-race noise and better reflects current competitive level.
- **`driver_circuit_top10_rate_prior` (interaction + historical rate):** Some drivers consistently outperform at specific circuits; this feature encodes driver-track fit.
- **`constructor_tier_prev_season` (categorical encoding):** Team performance tiers from the previous season proxy car competitiveness known before a new season race.


## 3. Temporal Split (same boundaries as Lab 1)
- Train: 2022–2023 full seasons
- Validation: 2024 rounds 1–12
- Test (sealed): 2024 rounds 13+


In [4]:
train = results[results["season"].isin([2022, 2023])].copy()
val = results[(results["season"] == 2024) & (results["round"] <= 12)].copy()
test = results[(results["season"] == 2024) & (results["round"] > 12)].copy()

assert train["date"].max() < val["date"].min(), "Train/Val leakage: overlapping dates"
assert val["date"].max() < test["date"].min(), "Val/Test leakage: overlapping dates"
assert set(zip(train["season"], train["round"])).isdisjoint(set(zip(val["season"], val["round"]))), "Train/Val race overlap"
assert set(zip(val["season"], val["round"])).isdisjoint(set(zip(test["season"], test["round"]))), "Val/Test race overlap"

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print(f"Val date range: {val['date'].min().date()} to {val['date'].max().date()}")
print(f"Validation top-10 rate: {val['top10_finish'].mean():.3f}")


Train: 880 | Val: 239 | Test: 240
Val date range: 2024-03-02 to 2024-07-07
Validation top-10 rate: 0.502


## 4. Leakage Guard Checklist (10 items)
| # | Check | Status | Evidence |
|---|---|---|---|
| 1 | No feature uses information from after the race being predicted | PASS | All engineered history features use `shift(1)` and/or previous-season mapping |
| 2 | Train/val/test split is strictly temporal | PASS | Date/race overlap assertions above |
| 3 | No direct target encoding (`position`, `points`, etc.) in model features | PASS | Model uses engineered pre-race proxies only |
| 4 | Encoder/imputer fit on training only | PASS | Train medians + train dummy columns reused on val |
| 5 | Rolling/lag features exclude current row | PASS | Explicit `shift(1)` in lag/rolling features |
| 6 | Group features computed with training-safe temporal logic | PASS | Driver/circuit history is prior-only via chronological expansion |
| 7 | Hyperparameters not tuned on test set | PASS | Fixed LogisticRegression config, no test tuning |
| 8 | No test-set data snooping | PASS | Test split created and never used in evaluation selection |
| 9 | `RANDOM_SEED = 414` for stochastic components | PASS | Seed set globally; sklearn random_state fixed |
| 10 | Notebook reproducible top-to-bottom | PASS* | Designed for restart-run-all (*requires API availability) |


## 5. Baselines + Improved Model
Primary metric remains **F1 (Top-10 class)** to match Lab 1.


In [5]:
def safe_roc_auc(y_true, y_score):
    try:
        return roc_auc_score(y_true, y_score)
    except ValueError:
        return np.nan


def compute_metrics(y_true, y_pred, y_score=None):
    if y_score is None:
        y_score = y_pred
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": safe_roc_auc(y_true, y_score),
    }


y_train = train["top10_finish"].astype(int)
y_val = val["top10_finish"].astype(int)

# Baseline 1: Majority class
X_train_grid = train[["grid"]]
X_val_grid = val[["grid"]]

majority = DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)
majority.fit(X_train_grid, y_train)
y_pred_majority = majority.predict(X_val_grid)
majority_score = np.full(len(val), y_train.mean())

# Baseline 2: Domain heuristic (exact Lab 1 rule)
y_pred_heuristic = ((val["grid"] >= 1) & (val["grid"] <= 10)).astype(int)
heuristic_score = y_pred_heuristic.astype(float)

# Baseline 3: Prior-period baseline
# Rule: avg finish in last 3 prior races <= 10 => predict Top-10
y_pred_prior = (val["avg_finish_pos_last3_prior"] <= 10).fillna(False).astype(int)
prior_score = y_pred_prior.astype(float)

# Improved model feature set
num_cols = [
    "grid",
    "prev_finish_pos",
    "avg_finish_pos_last3_prior",
    "driver_circuit_top10_rate_prior",
]
cat_cols = ["constructor_tier_prev_season"]

X_train_raw = train[num_cols + cat_cols].copy()
X_val_raw = val[num_cols + cat_cols].copy()

# Imputation fit on train only
train_medians = X_train_raw[num_cols].median()
X_train_raw[num_cols] = X_train_raw[num_cols].fillna(train_medians)
X_val_raw[num_cols] = X_val_raw[num_cols].fillna(train_medians)

X_train_raw[cat_cols] = X_train_raw[cat_cols].fillna("unknown")
X_val_raw[cat_cols] = X_val_raw[cat_cols].fillna("unknown")

# One-hot encoding with alignment
X_train_model = pd.get_dummies(X_train_raw, columns=cat_cols, drop_first=False)
X_val_model = pd.get_dummies(X_val_raw, columns=cat_cols, drop_first=False)
X_val_model = X_val_model.reindex(columns=X_train_model.columns, fill_value=0)

assert X_train_model.isna().sum().sum() == 0
assert X_val_model.isna().sum().sum() == 0

model = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)
model.fit(X_train_model, y_train)
y_pred_model = model.predict(X_val_model)
y_score_model = model.predict_proba(X_val_model)[:, 1]

rows = [
    ("Majority class (Lab 1)", compute_metrics(y_val, y_pred_majority, majority_score)),
    ("Domain heuristic (Lab 1)", compute_metrics(y_val, y_pred_heuristic, heuristic_score)),
    ("Prior-period baseline", compute_metrics(y_val, y_pred_prior, prior_score)),
    ("Lab 2 model (LogReg)", compute_metrics(y_val, y_pred_model, y_score_model)),
]

comparison_df = pd.DataFrame({"Model / Baseline": [r[0] for r in rows]})
for metric in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]:
    comparison_df[metric] = [r[1][metric] for r in rows]

comparison_df


,Model / Baseline,Accuracy,Precision,Recall,F1,ROC-AUC
0,Majority class (Lab 1),0.497908,0.000000,0.000000,0.000000,0.500000
1,Domain heuristic (Lab 1),0.857741,0.858333,0.858333,0.858333,0.857738
2,Prior-period baseline,0.778243,0.819048,0.716667,0.764444,0.778501
3,Lab 2 model (LogReg),0.824268,0.836207,0.808333,0.822034,0.893172


In [6]:
# Rounded display for reporting
comparison_rounded = comparison_df.copy()
for col in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]:
    comparison_rounded[col] = comparison_rounded[col].round(4)

print(comparison_rounded.to_string(index=False))

# Save machine-readable outputs for markdown consistency
out_dir = Path("labs/lab_02")
if not out_dir.exists():
    out_dir = Path(".")

comparison_rounded.to_csv(out_dir / "lab02_metrics_table.csv", index=False)
print(f"Saved metrics CSV to: {out_dir / 'lab02_metrics_table.csv'}")


        Model / Baseline  Accuracy  Precision  Recall     F1  ROC-AUC
  Majority class (Lab 1)    0.4979     0.0000  0.0000 0.0000   0.5000
Domain heuristic (Lab 1)    0.8577     0.8583  0.8583 0.8583   0.8577
   Prior-period baseline    0.7782     0.8190  0.7167 0.7644   0.7785
    Lab 2 model (LogReg)    0.8243     0.8362  0.8083 0.8220   0.8932
Saved metrics CSV to: lab02_metrics_table.csv


## 6. Result Interpretation (baseline gap)


In [7]:
f1_majority = comparison_df.loc[comparison_df["Model / Baseline"] == "Majority class (Lab 1)", "F1"].iloc[0]
f1_heuristic = comparison_df.loc[comparison_df["Model / Baseline"] == "Domain heuristic (Lab 1)", "F1"].iloc[0]
f1_prior = comparison_df.loc[comparison_df["Model / Baseline"] == "Prior-period baseline", "F1"].iloc[0]
f1_model = comparison_df.loc[comparison_df["Model / Baseline"] == "Lab 2 model (LogReg)", "F1"].iloc[0]

hardest_name = max(
    [
        ("Domain heuristic (Lab 1)", f1_heuristic),
        ("Prior-period baseline", f1_prior),
        ("Majority class (Lab 1)", f1_majority),
    ],
    key=lambda x: x[1],
)[0]

print("Primary metric: F1 (Top-10 class)")
print(f"Lab 2 model F1: {f1_model:.4f}")
print(f"Majority baseline F1: {f1_majority:.4f} (gap = {f1_model - f1_majority:+.4f})")
print(f"Heuristic baseline F1: {f1_heuristic:.4f} (gap = {f1_model - f1_heuristic:+.4f})")
print(f"Prior-period baseline F1: {f1_prior:.4f} (gap = {f1_model - f1_prior:+.4f})")
print(f"Hardest baseline to beat by F1: {hardest_name}")

if f1_model > f1_heuristic:
    print("Interpretation: The engineered-feature model beats the Lab 1 heuristic baseline on the primary metric.")
else:
    print("Interpretation: The engineered-feature model does NOT beat the Lab 1 heuristic baseline. This is valid and indicates the simple rule remains very strong.")


Primary metric: F1 (Top-10 class)
Lab 2 model F1: 0.8220
Majority baseline F1: 0.0000 (gap = +0.8220)
Heuristic baseline F1: 0.8583 (gap = -0.0363)
Prior-period baseline F1: 0.7644 (gap = +0.0576)
Hardest baseline to beat by F1: Domain heuristic (Lab 1)
Interpretation: The engineered-feature model does NOT beat the Lab 1 heuristic baseline. This is valid and indicates the simple rule remains very strong.


## 7. Error Analysis — Top 3 Failure Modes
We identify concrete failure slices from validation predictions.


In [8]:
val_analysis = val[[
    "season", "round", "race_name", "driver", "driver_code", "constructor", "circuit",
    "grid", "position", "status", "top10_finish",
    "prev_finish_pos", "avg_finish_pos_last3_prior", "driver_circuit_top10_rate_prior",
    "constructor_tier_prev_season"
]].copy()

val_analysis["pred"] = y_pred_model
val_analysis["pred_proba_top10"] = y_score_model
val_analysis["error_type"] = np.where(
    (val_analysis["pred"] == 1) & (val_analysis["top10_finish"] == 0),
    "FP",
    np.where((val_analysis["pred"] == 0) & (val_analysis["top10_finish"] == 1), "FN", "OK"),
)

# Failure mode 1: FP from strong grid starts that collapse (often incidents/DNFs)
mode1 = val_analysis[(val_analysis["error_type"] == "FP") & (val_analysis["grid"] <= 10)]    .sort_values(["pred_proba_top10"], ascending=False)    .head(5)

# Failure mode 2: FN from deep-grid chargers
mode2 = val_analysis[(val_analysis["error_type"] == "FN") & (val_analysis["grid"] > 10)]    .sort_values(["grid"], ascending=False)    .head(5)

# Failure mode 3: Circuit interaction misses (driver-circuit prior low, actual top10)
mode3 = val_analysis[(val_analysis["error_type"] == "FN") & (val_analysis["driver_circuit_top10_rate_prior"].fillna(0) < 0.2)]    .sort_values(["pred_proba_top10"], ascending=True)    .head(5)

print("Failure Mode 1 — FP: Top-grid starters predicted Top-10 but finished outside Top-10")
display(mode1[["season", "round", "race_name", "driver_code", "constructor", "grid", "position", "status", "pred_proba_top10"]])

print("Failure Mode 2 — FN: Deep-grid chargers predicted outside Top-10 but actually finished Top-10")
display(mode2[["season", "round", "race_name", "driver_code", "constructor", "grid", "position", "status", "pred_proba_top10"]])

print("Failure Mode 3 — FN: Low driver-circuit prior rate but actual Top-10")
display(mode3[["season", "round", "race_name", "driver_code", "circuit", "driver_circuit_top10_rate_prior", "grid", "position", "pred_proba_top10"]])


Failure Mode 1 — FP: Top-grid starters predicted Top-10 but finished outside Top-10


,season,round,race_name,driver_code,constructor,grid,position,status,pred_proba_top10
928,2024,3,Australian Grand Prix,VER,red_bull,1,19,Retired,0.956658
1113,2024,12,British Grand Prix,RUS,mercedes,1,19,Retired,0.944719
1088,2024,11,Austrian Grand Prix,NOR,mclaren,2,20,Lapped,0.916979
934,2024,3,Australian Grand Prix,RUS,mercedes,7,17,Retired,0.856814
1086,2024,11,Austrian Grand Prix,LEC,ferrari,6,11,Finished,0.810803


Failure Mode 2 — FN: Deep-grid chargers predicted outside Top-10 but actually finished Top-10


,season,round,race_name,driver_code,constructor,grid,position,status,pred_proba_top10
963,2024,5,Chinese Grand Prix,HAM,mercedes,18,9,Finished,0.342091
1049,2024,9,Canadian Grand Prix,OCO,alpine,18,10,Finished,0.222837
925,2024,3,Australian Grand Prix,HUL,haas,16,9,Finished,0.121312
1042,2024,9,Canadian Grand Prix,GAS,alpine,15,9,Finished,0.271928
906,2024,2,Saudi Arabian Grand Prix,HUL,haas,15,10,Finished,0.078649


Failure Mode 3 — FN: Low driver-circuit prior rate but actual Top-10


,season,round,race_name,driver_code,circuit,driver_circuit_top10_rate_prior,grid,position,pred_proba_top10
906,2024,2,Saudi Arabian Grand Prix,HUL,jeddah,0.0,15,10,0.078649
926,2024,3,Australian Grand Prix,MAG,albert_park,0.0,14,10,0.112785
1117,2024,12,British Grand Prix,TSU,silverstone,0.0,13,10,0.208760
1084,2024,11,Austrian Grand Prix,HUL,red_bull_ring,0.0,9,6,0.221330
1042,2024,9,Canadian Grand Prix,GAS,villeneuve,0.0,15,9,0.271928


### Error Analysis Hypotheses + Next Steps
1. **Top-grid collapse FPs (often incidents/reliability):** The model over-trusts grid/history features and cannot see race-day shocks (DNF, damage, penalties).  
   **Next try:** add pre-race reliability proxies (constructor/driver DNF rates from prior races).

2. **Deep-grid charger FNs:** The model underestimates comeback potential for strong pace drivers starting outside Top-10.  
   **Next try:** add qualifying gap-to-pole or prior race pace trend features (still pre-race).

3. **Driver-circuit history miss FNs:** Sparse historical interaction data can bias against rare but real improvements at specific circuits.  
   **Next try:** smooth driver-circuit priors with hierarchical blending (driver global prior + circuit prior + interaction prior).


## 8. Final Notes
- `RANDOM_SEED = 414` is used in all stochastic model components.
- Test split is explicitly sealed and unused for model selection.
- If this model fails to beat the strongest baseline, that is still an informative result and drives next feature iterations.
